# 09b - Map Shared Taxonomy Back to Tessellation Cells

Split out of `09_clustering.ipynb` so that notebook can run once globally
on pooled multi-region data while this one runs **per region**, mapping the
shared taxonomy (`cluster_mapping.parquet` + `cdata.parquet`) back onto that
region's own local tessellation cells (`06_morphotopes/tessellation_labels_postprocessed.parquet`).

For a single-city run (no `GLOBAL_HIER_DIR` override) this behaves exactly
like the removed cell in `09_clustering.ipynb` — `GLOBAL_HIER_DIR` defaults to
the same `OUTPUT_ROOT` the rest of the notebook uses.

After mapping, `cluster_mapping.parquet` and `cdata.parquet` are also copied
into this region's local `09_hierarchy/` folder so notebooks 10-11 (noise
assignment, export) can keep reading `HIER_DIR` unmodified, whether or not
this is a chunked run.

In [ ]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import shutil
import pandas as pd
from pathlib import Path

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
MORPH_DIR = Path(OUTPUT_ROOT) / "06_morphotopes"
HIER_DIR = Path(OUTPUT_ROOT) / "09_hierarchy"
HIER_DIR.mkdir(parents=True, exist_ok=True)

# Where the shared taxonomy (cluster_mapping.parquet + cdata.parquet) lives.
# Defaults to this same OUTPUT_ROOT so single-city runs are unaffected;
# region_pipeline.py points this at the pooled global root for chunked runs.
GLOBAL_HIER_DIR = Path(os.environ.get("GLOBAL_HIER_DIR", OUTPUT_ROOT)) / "09_hierarchy"
print(f"Local HIER_DIR: {HIER_DIR}")
print(f"Global HIER_DIR: {GLOBAL_HIER_DIR}")

## Load shared taxonomy + this region's local morphotope labels

In [ ]:
cluster_mapping = pd.read_parquet(GLOBAL_HIER_DIR / "cluster_mapping.parquet")
cdata_median = pd.read_parquet(GLOBAL_HIER_DIR / "cdata.parquet")
morph_labels = pd.read_parquet(
    MORPH_DIR / "tessellation_labels_postprocessed.parquet"
)["morphotope_label"]

print(f"Shared taxonomy: {cluster_mapping.shape}")
print(f"Local morphotope labels: {morph_labels.shape}")

## Map cluster labels back to tessellation cells

In [ ]:
# Map finest level back to every tessellation cell
finest = cluster_mapping['level_1']  # level_1 = finest
morph_to_cluster = finest.to_dict()

# Assign to tessellation cells (non-noise only for now; noise handled in notebook 10)
tess_clusters = morph_labels.map(morph_to_cluster)
tess_clusters.name = 'final'

# Also map all hierarchy levels
for col in cluster_mapping.columns:
    tess_clusters_level = morph_labels.map(cluster_mapping[col].to_dict())
    tess_clusters = pd.concat([tess_clusters, tess_clusters_level.rename(col)], axis=1)

tess_clusters.to_parquet(HIER_DIR / "tessellation_clusters.parquet")
print(f"Mapped {tess_clusters['final'].notna().sum():,} cells to clusters")
print(f"  Unmapped (noise): {tess_clusters['final'].isna().sum():,}")

# Copy the shared taxonomy into this region's local HIER_DIR too, so
# notebooks 10 (noise) and 11 (export) can keep reading HIER_DIR unmodified
# regardless of whether GLOBAL_HIER_DIR pointed elsewhere.
if GLOBAL_HIER_DIR.resolve() != HIER_DIR.resolve():
    shutil.copy(GLOBAL_HIER_DIR / "cluster_mapping.parquet", HIER_DIR / "cluster_mapping.parquet")
    shutil.copy(GLOBAL_HIER_DIR / "cdata.parquet", HIER_DIR / "cdata.parquet")
else:
    cdata_median.to_parquet(HIER_DIR / "cdata.parquet")
print(f"Cluster centroid data available at {HIER_DIR / 'cdata.parquet'}")